In [12]:
import sys
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.recommend_model import (
    LinearRecommender,
    RandomForestRecommender,
    CatBoostRecommender,
    NCFRecommender
)

In [13]:
df = pd.read_csv("../data/recommender_training.csv")

# --- split user ---
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)

# 2) Val 15%, Test 15%
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

Train: (82555, 12)
Val  : (17690, 12)
Test : (17691, 12)


In [14]:
# cat = CatBoostRecommender()
# cat.train(train_df, val_df)
# cat.save("../models/catboost_rec.pkl")

# ncf = NCFRecommender()
# ncf.train(train_df)
# ncf.save("../models/ncf_rec.pkl")

# linear = LinearRecommender()
# linear.train(train_df)
# linear.save("../models/linearregression_rec.pkl")

# rf = RandomForestRecommender()
# rf.train(train_df)
# rf.save("../models/randomforest_rec.pkl")


In [15]:
def evaluate_regression_model(recommender, df):
    """
    Đánh giá regression thuần (label prediction)
    """
    df = recommender.feature_engineering(df)

    if "CatBoost" in recommender.model_name:
        for col in recommender.cat_features:
            df[col] = df[col].astype(str).fillna("unknown")

        X = df[recommender.features]

    else:
        df = recommender.transform_encoders(df)
        X = df[recommender.features]
        if recommender.scaler:
            X = recommender.scaler.transform(X)

    y = df["label"]

    return recommender.evaluate(X, y)


In [16]:
def build_ground_truth(df, top_n=5):
    return (
        df.sort_values("label", ascending=False)
          .groupby("user_id")
          .head(top_n)
          .groupby("user_id")["poi_id"]
          .apply(set)
          .to_dict()
    )

In [17]:
def precision_at_k(recommended, relevant, k):
    rec_k = recommended[:k]
    return len(set(rec_k) & relevant) / k

def recall_at_k(recommended, relevant, k):
    if not relevant:
        return 0.0
    return len(set(recommended[:k]) & relevant) / len(relevant)

In [18]:
def evaluate_recommender(recommender, df_test, poi_df, k=10):
    gt = build_ground_truth(df_test, top_n=k)

    precisions, recalls = [], []
    skipped = 0

    for user_id, relevant_pois in gt.items():

        # --- SKIP cold-start user cho NCF ---
        if recommender.model_name == "NCF":
            if user_id not in recommender.user_encoder.classes_:
                skipped += 1
                continue

        city, user_type, price = user_id.split("_")

        candidate_ids = df_test[df_test["user_id"] == user_id]["poi_id"]
        candidate_pois = poi_df[poi_df["poi_id"].isin(candidate_ids)]

        if candidate_pois.empty:
            continue

        recs = recommender.recommend(
            df_raw=candidate_pois,
            city=city,
            user_type=user_type,
            user_price=int(price),
            top_k=k
        )

        if recs.empty:
            continue

        recommended_pois = recs["poi_id"].tolist()

        precisions.append(len(set(recommended_pois) & relevant_pois) / k)
        recalls.append(len(set(recommended_pois) & relevant_pois) / max(1, len(relevant_pois)))

    return {
        f"Precision@{k}": np.mean(precisions) if precisions else 0.0,
        f"Recall@{k}": np.mean(recalls) if recalls else 0.0
    }

In [19]:
poi_df = pd.read_csv("../data/POI.csv")

cat = CatBoostRecommender()
cat.load("../models/catboost_rec.pkl")

ncf = NCFRecommender()
ncf.load("../models/ncf_rec.pkl")

linear = LinearRecommender()
linear.load("../models/linearregression_rec.pkl")

rf = RandomForestRecommender()
rf.load("../models/randomforest_rec.pkl")

print("=== REGRESSION METRICS ===")
print("Linear:", evaluate_regression_model(linear, test_df))
print("RandomForest:", evaluate_regression_model(rf, test_df))
print("CatBoost:", evaluate_regression_model(cat, test_df))


print("=== RANKING METRICS ===")
print("Linear:", evaluate_recommender(linear, test_df, poi_df))
print("RandomForest:", evaluate_recommender(rf, test_df, poi_df))
print("CatBoost:", evaluate_recommender(cat, test_df, poi_df))
print("NCF:", evaluate_recommender(ncf, test_df, poi_df))


✓ [CatBoost] Model loaded.
✓ [NCF] Model loaded
✓ [LinearRegression] Model loaded.
✓ [RandomForest] Model loaded.
=== REGRESSION METRICS ===
Linear: {'MAE': 0.06916510559448816, 'RMSE': 0.0919374789413469, 'R2': 0.4448348926119229}
RandomForest: {'MAE': 0.010791409728861107, 'RMSE': 0.020679316264129582, 'R2': 0.9719127427584046}
CatBoost: {'MAE': 0.0013729249125690784, 'RMSE': 0.004280445185883736, 'R2': 0.9987965865494862}
=== RANKING METRICS ===
Linear: {'Precision@10': 0.41466314398943194, 'Recall@10': 0.7163804491413474}
RandomForest: {'Precision@10': 0.5848084544253633, 'Recall@10': 0.8865257595772787}
CatBoost: {'Precision@10': 0.6586525759577279, 'Recall@10': 0.9603698811096433}
NCF: {'Precision@10': 0.4403458213256484, 'Recall@10': 0.6877521613832852}


In [20]:
# ===== FINAL TRAIN DATA (train + val) =====
full_train_df = pd.concat([train_df, val_df], ignore_index=True)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Final Train (train+val):", full_train_df.shape)

final_cat = CatBoostRecommender()

# Train trên toàn bộ train + val
final_cat.train(full_train_df)

# Save model FINAL
final_cat.save("../models/catboost_rec_final.pkl")

Train: (82555, 12)
Val  : (17690, 12)
Final Train (train+val): (100245, 12)
[CatBoost] Training finished.
✓ [CatBoost] Model saved to ../models/catboost_rec_final.pkl
